<div class="jumbotron" style="background:WhiteSmoke">
  <br>
    <hr style="text-align:center">
      <h1 class="title" style="text-align:center;color:RoyalBlue">Checkpoint 12</h1>
      <h2 class="subtitle" style="text-align:center;color:Black">Advanced Navigation</h2>
    <hr style="text-align:center">
  <br>
</div>

<div class="separator-primary" style="height:1.6em;width:100%;background:LightGrey">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- Summary -</b></p>
</div>

The purpose of this project is to design the **Navigation System** of a mobile robot that must work in a warehouse. 

In the first part of the project, **Checkpoint 11**, you had already set up and configured the navigation system of the **RB1** mobile robot.

In this second part of the project, you will use the **Simple Commander API**, that interacts with the **Nav2** system, in order to create some navigation routes for the robot.

For this project you will use a simulation of the warehouse in which the **RB1** robot will work. To launch it, run the following commands:

<div class="execute" style="width:14.0em;height:1.5em;border-radius:0.8em;background:Silver">
  <p class="executetext" style="text-align:center;color:Black">
    <i class="fa fa-terminal" style="font-size:1.5em"></i>
    &nbsp;
    <b>Execute in Terminal</b>
  </p>
</div>

In [ ]:
source ~/sim_ws/install/setup.bash
ros2 launch the_construct_office_gazebo warehouse_rb1.launch.xml

<div class="separator-primary" style="height:1.6em;width:100%;background:LightGrey">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- End of Summary -</b></p>
</div>

<div class="section" style="height:1.8em;background:WhiteSmoke">
  <h2 class="section-title" style="text-align:center">
    <span class="section-title-secondary" style="text-align:center;color:Black">Preparing the Environment</span>
  </h2>
</div>

<div class="separator-primary" style="height:1.6em;width:100%;background:Tomato">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- Important Notes -</b></p>
</div>

## Setting the Git Environment

As already mentioned, in order to proceed with this Checkpoint you must have correctly finished the **Checkpoint 11**. So, the first thing you have to do is to download the repository that you created in **Checkpoint 11**, which contains the packages for launching the **RB1** navigation system. Download the repository inside your ***`~/ros2_ws/src`*** directory.

<div class="execute" style="width:14.0em;height:1.5em;border-radius:0.8em;background:Silver">
  <p class="executetext" style="text-align:center;color:Black">
    <i class="fa fa-terminal" style="font-size:1.5em"></i>
    &nbsp;
    <b>Execute in Terminal</b>
  </p>
</div>

In [ ]:
cd ~/ros2_ws/src
git clone <your_repository_url>

Once you have downloaded your repository, compile the packages and make sure you can launch the **RB1** navigation system correctly.

<div class="separator-primary" style="height:1.6em;width:100%;background:Tomato">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- End of Important Notes -</b></p>
</div>

<div class="section" style="height:1.8em;background:WhiteSmoke">
  <h2 class="section-title" style="text-align:center">
    <span class="section-title-primary" style="text-align:center;color:RoyalBlue">Task 1</span>
    &nbsp;&nbsp;&nbsp;
    <span class="section-title-secondary" style="text-align:center;color:Black">Simple Commander API</span>
  </h2>
</div>

<div class="section" style="height:1.8em;background:WhiteSmoke">
  <h2 class="section-title" style="text-align:center">
    <span class="section-title-primary" style="text-align:center;color:RoyalBlue">1.1</span>
    &nbsp;&nbsp;&nbsp;
    <span class="section-title-secondary" style="text-align:center;color:Black">Create a ROS2 Node to Navigate</span>
  </h2>
</div>

Now that the **Navigation System** is working, and the **RB1** robot can navigate autonomously, let's make an application that uses the navigation skill to perform actual tasks using the **Simple Commander API**.

The goal is the following:

- Once the application is launched, it has to localize the robot in the ***init_position***.
- Then, make the robot go underneath the shelf that will be near the ***loading_position*** and carry it.
- Afterwards, move the shelf to the ***shipping_position*** while avoiding the cones area completely.
- Finally, unload the robot shelf and return to the ***init_position***.

Let's start by defining a simple route for the robot.

1. Inside ***`warehouse_project`***, create a new package named ***`nav2_apps`***, where you'll place the scripts for this section.

2. Inside this package, create a new **Python** script named ***`move_shelf_to_ship.py`*** that will do the following:
    - The program will use the **Simple Commander API** to localize the robot in the ***init_position***.
    - The program will send a first goal to the robot using the ***`NavigateToPose`*** action.
    - The first goal will be located inside the ***loading_position*** (see image below).

<center><img src="images/lab_map2.png" width="750"/></center>

3. Once the robot is at the ***loading_position*** it has to get underneath the shelf and activate the elevator to carry it.

<div class="separator-primary" style="height:1.6em;width:100%;background:LightBlue">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- Notes -</b></p>
</div>

- To move the robot underneath the shelf and lift it you will **not** be able to use navigation (since it will detect the shelf as an obstacle). For this, you can recycle the code you created for **Checkpoint 9**, for both: the simulation and the real robot.

- Remember that when it has loaded the shelf, the shape of the robot will be larger (because now the robot is carrying the shelf). So you need to change its shape (robot footprint) in **Nav2** in real time (while navigation is still running) to prevent the robot planning over places where the robot+shelf cannot fit.

<div class="separator-primary" style="height:1.6em;width:100%;background:LightBlue">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- End of Notes -</b></p>
</div>

For the next step, you need to send the robot to the ***shipping_position***.

But, as you may have noticed, there are some cones in the middle of the warehouse, which might interfere with the route of the robot and cause some difficulties. So let's make sure the robot avoids these cones!

4. Create a Costmap Filter that adds a **Keepout Mask**. This Keepout Mask has to make the robot avoid the area where the cones are placed (see image below). Save this new map with the keepout mask as ***`warehouse_map_keepout_sim`***.

<center><img src="images/keepout_mask.png" width="750"/></center>

Then, you are ready to send the robot with the shelf to the ***shipping_position*** while avoiding the cones area.

8. Add a new call to the **API** to move the robot+shelf to the ***shipping_position***

9. Once at that position, make the robot move down the elevator and get out from underneath the shelf. Put back the shape parameter for navigation to the original size of the robot.

10. Finally, send another goal to the ***init_position***.

The robot is now ready to move another shelf.

<div class="separator-primary" style="height:1.6em;width:100%;background:GoldenRod">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- Grading Guide -</b></p>
</div>

### Task 1.1 - Navigation using Simple Commander API in Simulation

Start **Nav2** with the following commands:

<div class="execute" style="width:14.0em;height:1.5em;border-radius:0.8em;background:Silver">
  <p class="executetext" style="text-align:center;color:Black">
    <i class="fa fa-terminal" style="font-size:1.5em"></i>
    &nbsp;
    <b>Execute in Terminal</b>
  </p>
</div>

In [ ]:
ros2 launch localization_server localization.launch.py map_file:=warehouse_map_keepout_sim.yaml

In [ ]:
ros2 launch path_planner_server pathplanner.launch.py use_sim_time:=True

- When the **Navigation** programs are started, **RViz** also gets launched with a pre-defined configuration from a ***`.rviz`*** config file - **1.0 point**

- **RViz** loads with the required display elements for visualizing the navigation task such as Map, RobotModel, TF, LaserScan, Odometry, ParticleCloud, PoseWithCovariance, Global & Local Costmaps, Global & Local Paths, Robot Footprint, etc - **1.5 points**

- **RViz** loads with the proper view angle and the whole map can be visualized - **1.0 point**

- When the navigation system has started, the area with the cones appears with a Keepout Mask on the **RViz** map - **1.0 point**

Start the ***`move_shelf_to_ship.py`*** script with the following command:

In [ ]:
python3 ~/ros2_ws/src/warehouse_project/nav2_apps/scripts/move_shelf_to_ship.py

- When executing the ***`move_shelf_to_ship.py`*** script, the robot is able to properly localize itself and arrive at the ***`loading_position`*** - **1.0 point**

- The robot is able to get underneath the shelf and load it, then change its footprint for navigation - **1.5 points**

- The robot is able to navigate to the ***`shipping_position`*** while carrying the shelf - **1.0 point**

- The robot unloads the shelf and gets out of it, then changes its footprint back to initial size and returns to the ***`init_position`*** - **2.0 points**

<div class="separator-primary" style="height:1.6em;width:100%;background:GoldenRod">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- End of Grading Guide -</b></p>
</div>

<div class="section" style="height:1.8em;background:WhiteSmoke">
  <h2 class="section-title" style="text-align:center">
    <span class="section-title-primary" style="text-align:center;color:RoyalBlue">1.2</span>
    &nbsp;&nbsp;&nbsp;
    <span class="section-title-secondary" style="text-align:center;color:Black">Test Everything in the Real Robot Lab</span>
  </h2>
</div>

Now it is time that you test your program with the real robot.

1. Book a 1-hour session of the **RB1** real robot lab.
2. On the date and time selected, open this rosject and connect to the real robot.
3. Create a new map with the keepout mask for the real robot lab and name it ***`warehouse_map_keepout_real`***.
4. Execute your program and see if the results are the same.

<div class="separator-primary" style="height:1.6em;width:100%;background:LightBlue">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- Notes -</b></p>
</div>

<p><b><span style="color:Red">Note: </span></b>When you are finished working with the real robot, please remember to move the robot to its initial position.</p>

<div class="separator-primary" style="height:1.6em;width:100%;background:LightBlue">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- End of Notes -</b></p>
</div>

<div class="separator-primary" style="height:1.6em;width:100%;background:GoldenRod">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- Grading Guide -</b></p>
</div>

### Task 1.2 - Navigation using Simple Commander API in Real Robot

Start **Nav2** with the following commands:

<div class="execute" style="width:14.0em;height:1.5em;border-radius:0.8em;background:Silver">
  <p class="executetext" style="text-align:center;color:Black">
    <i class="fa fa-terminal" style="font-size:1.5em"></i>
    &nbsp;
    <b>Execute in Terminal</b>
  </p>
</div>

In [ ]:
ros2 launch localization_server localization.launch.py map_file:=warehouse_map_keepout_real.yaml

In [ ]:
ros2 launch path_planner_server pathplanner.launch.py use_sim_time:=False

- When the **Navigation** programs are started, **RViz** also gets launched with a pre-defined configuration from a ***`.rviz`*** config file - **1.0 point**

- **RViz** loads with the required display elements for visualizing the navigation task such as Map, RobotModel, TF, LaserScan, Odometry, ParticleCloud, PoseWithCovariance, Global & Local Costmaps, Global & Local Paths, Robot Footprint, etc - **1.5 points**

- **RViz** loads with the proper view angle and the whole map can be visualized - **1.0 point**

- When the navigation system has started, the area with the cones appears with a Keepout Mask on the **RViz** map - **1.0 point**

Start the ***`move_shelf_to_ship_real.py`*** script with the following command:

In [ ]:
python3 ~/ros2_ws/src/warehouse_project/nav2_apps/scripts/move_shelf_to_ship_real.py

- When executing the ***`move_shelf_to_ship_real.py`*** script, the robot is able to properly localize itself and arrive at the ***`loading_position`*** - **1.0 point**

- The robot is able to get underneath the shelf and load it, then change its footprint for navigation - **1.5 points**

- The robot is able to navigate to the ***`shipping_position`*** while carrying the shelf - **1.0 point**

- The robot unloads the shelf and gets out of it, then changes its footprint back to initial size and returns to the ***`init_position`*** - **2.0 points**

<div class="separator-primary" style="height:1.6em;width:100%;background:GoldenRod">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- End of Grading Guide -</b></p>
</div>

<div class="section" style="height:1.8em;background:WhiteSmoke">
  <h2 class="section-title" style="text-align:center">
    <span class="section-title-primary" style="text-align:center;color:RoyalBlue">Appendix</span>
    &nbsp;&nbsp;&nbsp;
    <span class="section-title-secondary" style="text-align:center;color:Black">Load / Unload Shelf</span>
  </h2>
</div>

To load the shelf with the RB1, you need to publish to the **`/elevator_up`** and **`/elevator_down`** topics, both in the simulated environment and on the real robot.

Before raising the elevator, ensure that the robot is positioned underneath the shelf. Once the robot is correctly aligned, enter the following command in a new terminal to elevate the shelf:

<div class="execute" style="width:14.0em;height:1.5em;border-radius:0.8em;background:Silver">
  <p class="executetext" style="text-align:center;color:Black">
    <i class="fa fa-terminal" style="font-size:1.5em"></i>
    &nbsp;
    <b>Execute in Terminal</b>
  </p>
</div>

In [ ]:
ros2 topic pub /elevator_up std_msgs/msg/String --once

Now, when you move the RB1, the cart should move along with it.

To lower the shelf, simply publish to the `/elevator_down` topic:

<div class="execute" style="width:14.0em;height:1.5em;border-radius:0.8em;background:Silver">
  <p class="executetext" style="text-align:center;color:Black">
    <i class="fa fa-terminal" style="font-size:1.5em"></i>
    &nbsp;
    <b>Execute in Terminal</b>
  </p>
</div>

In [ ]:
ros2 topic pub /elevator_down std_msgs/msg/String --once

<div class="separator-primary" style="height:1.6em;width:100%;background:Tomato">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- IMPORTANT NOTES -</b></p>
</div>

## REAL ROBOT

When working with the real robot, there are three main considerations to keep in mind:

1. **Cart Orientation for Leg Detection**: When performing leg detection using laser intensity values, ensure the cart is properly oriented toward the robot. The correct orientation is shown in the image below:

<img src="images/tc-logo-cart-good.png" width="500" />

You can use the "The Construct" logo as a reference. If the logo appears upside down, the robot’s laser will not correctly detect the leg intensities. In that case, rotate the cart to face the correct direction.

2. **Positioning the Robot Underneath the Cart**: Make sure the robot is as centered as possible under the cart to ensure a successful attachment. If it is off-center, the cart may not attach properly.
   
**Good Centering**:

<img src="images/rb1-centered.png" width="500" />

**Bad Centering**:

<img src="images/rb1-not-centered.png" width="500" />

3. **Publishing Elevator Messages**: When raising or lowering the elevator on the real robot, you need to publish **2–3 consecutive messages** to the topic. Publishing a single message using `--once` may not be reliable, as the first message might not reach the robot. Sending multiple messages increases the chance of successful communication and elevator activation.

<img src="images/rb1-elevator-pub.gif" width="800" />

<div class="separator-primary" style="height:1.6em;width:100%;background:Tomato">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- END IMPORTANT NOTES -</b></p>
</div>

<div class="section" style="height:1.8em;background:WhiteSmoke">
  <h2 class="section-title" style="text-align:center">
    <span class="section-title-primary" style="text-align:center;color:RoyalBlue">Appendix</span>
    &nbsp;&nbsp;&nbsp;
    <span class="section-title-secondary" style="text-align:center;color:Black">Useful Commands</span>
  </h2>
</div>

You can send velocity commands to the mobile robot you are using by launching the following **ROS2** node.

<div class="execute" style="width:14.0em;height:1.5em;border-radius:0.8em;background:Silver">
  <p class="executetext" style="text-align:center;color:Black">
    <i class="fa fa-terminal" style="font-size:1.5em"></i>
    &nbsp;
    <b>Execute in Terminal</b>
  </p>
</div>

In [ ]:
ros2 run teleop_twist_keyboard teleop_twist_keyboard --ros-args --remap cmd_vel:=/diffbot_base_controller/cmd_vel_unstamped

**REMEMBER**, you need to have the focus on the terminal where you launched the program for the keys to take effect. You know you have correctly focused when the cursor starts blinking. 

<div class="separator-primary" style="height:1.6em;width:100%;background:DarkGray">
  <p class="separatortitle" style="text-align:center;color:Black"><b>- End of Notebook -</b></p>
</div>